# PHYS 142/242 — Assignment 3: MCMC, the 2D Ising Model, and Path-Integral Monte Carlo

**Name:** Ken Nguyen
**TSN:** 200288141
**Seed:** every random generator is seeded from `SEED = 200288141`; my $k$-th run uses `SEED + k` (table in Setup).

_TODO (at the end): one or two sentences on which parts I did (1A, 1B, 2A, and bonus 2B?) and that the notebook runs top to bottom (**Restart → Run All**)._

## Setup

**Units.** Problem 1: $J=k_B=1$, square lattice $L\times L$ with $L=20$ ($|\Lambda|=400$ spins), periodic boundaries.
Problem 2: $m=\hbar=\omega=1$, $\delta\tau=0.1$, $V(x)=\tfrac12x^2$.

**Time unit.** Both problems count MCMC time in **sweeps**: $|\Lambda|=400$ attempted spin flips (Problem 1) or $N=\beta/\delta\tau$ attempted slice updates (Problem 2). I record one measurement per sweep.

**Seeds** (assignment: integer $+\,k$ for the $k$-th run). Fixed run numbers, so any single run can be repeated at Oral Checkpoint 3:

| $k$ | run |
|---|---|
| 0, 1 | 1A: all-up start, random start |
| $100 + 10\,i_T + c$ | 1B: temperature index $i_T=0\ldots15$, chain $c$ |
| 200 | 2A(i): single-slice moves |
| 300 | 2B(i): both moves, $\beta=1$ |
| $400 + i_\beta$ | 2B(ii): $\beta=0.5,1,2,4$ |
| 900–999 | validation tests (not part of any reported result) |

**Shared error-analysis tools** (Lecture 16, Lab 12), used by both problems:
- autocorrelation $\hat\rho(t)=\hat c(t)/\hat c(0)$, with $\hat c(t)=\frac1N\sum_{i=1}^{N-t}(f_i-\bar f)(f_{i+t}-\bar f)$;
- integrated autocorrelation time $\hat\tau(M)=1+2\sum_{t=1}^{M}\hat\rho(t)$, cut at the smallest $M\ge5\hat\tau(M)$; trust it only if $N\gtrsim50\hat\tau$;
- blocking: block means of length $B$, $\sigma_{\bar f}(B)=\sqrt{\frac{1}{N_B(N_B-1)}\sum_b(F_b-\bar f)^2}$, doubled until it plateaus; $B=1$ is the naive error.

In [ ]:
import itertools
import json
import numpy as np
import matplotlib.pyplot as plt
from numba import njit

TSN = "200288141"
SEED = int(TSN)


def rng_for(k):
    """Random generator for my k-th run (seed = SEED + k, see the table above)."""
    return np.random.default_rng(SEED + k)


VALIDATION = []                                         # (test name, passed?) for the summary at the end


def check(name, ok, detail=""):
    """Record and print one validation test. Prints instead of asserting, so Run All always finishes."""
    VALIDATION.append((name, bool(ok)))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f"   ({detail})" if detail else ""))


# ---------- shared error-analysis tools (Lecture 16, Lab 12) ----------
def autocorrelation(f, max_lag):
    """Normalized autocorrelation rho(t), t = 0..max_lag (mean subtracted, divide by N)."""
    # TODO: c(t) = (1/N) sum_{i} (f_i - fbar)(f_{i+t} - fbar);  return c / c[0]
    raise NotImplementedError


def tau_int(f):
    """Integrated autocorrelation time 1 + 2 sum rho(t), automatic window: smallest M with M >= 5 tau(M)."""
    # TODO: loop M = 1, 2, ...; tau = 1 + 2 * rho[1:M+1].sum(); stop when M >= 5 * tau
    raise NotImplementedError


def naive_error(f):
    """Error on the mean if every sample were independent: s / sqrt(N)."""
    # TODO
    raise NotImplementedError


def blocking_curve(f, block_lengths):
    """Blocking error sigma(B) for each block length B (drop the leftover samples at the end)."""
    # TODO: block means F_b, then sqrt( sum (F_b - mean)^2 / (N_B (N_B - 1)) )
    raise NotImplementedError


def plot_blocking(ax, f, title):
    """Blocking curve with B = 1, 2, 4, ... up to N/32 blocks; mark the plateau I read off."""
    # TODO
    raise NotImplementedError

### Validation of the error-analysis tools
**Known answer** (Lab 10 checklist, item 1). Before trusting `tau_int` and blocking on my data, I run them on Lab 12's toy model, where everything is exact:
$$
x_t=r\,x_{t-1}+\sqrt{1-r^2}\,\epsilon_t,\qquad \rho(t)=r^t,\qquad \tau_\text{int}=1+2\sum_{t\ge1}r^t=\frac{1+r}{1-r}=19\ (r=0.9).
$$
The exact error of the mean is $\sigma_{\bar x}^2=\frac1N\big[1+2\sum_{k\ge1}(1-\tfrac kN)r^k\big]$. White noise ($r=0$) must give $\tau_\text{int}\approx1$ and a flat blocking curve.

| Test | Catches |
|---|---|
| `tau_int` = 19 ± 20 % at $r=0.9$ | missing factor 2, the $g/2$ convention, wrong window, no mean subtraction |
| `tau_int` ≈ 1 for white noise | an estimator that invents correlation |
| `naive_error` = $s/\sqrt N$; blocking at $B=1$ equals it | bookkeeping in `blocking_curve` |
| blocking plateau ($B=1024$, 128 blocks) = exact error ± 25 % | wrong normalization ($N_B$ vs $N_B-1$, missing square root) |

In [ ]:
def ar1(n, r, rng):
    """Lab 12's stationary AR(1) toy: known rho(t) = r^t, tau_int = (1 + r) / (1 - r)."""
    eps = rng.normal(size=n)
    x = np.empty(n)
    x[0] = eps[0]
    for t in range(1, n):
        x[t] = r * x[t - 1] + np.sqrt(1 - r**2) * eps[t]
    return x


def ar1_exact_error(n, r):
    k = np.arange(1, n)
    return np.sqrt((1 + 2 * np.sum((1 - k / n) * r**k)) / n)


N_TOY, R_TOY = 2**17, 0.9
x_toy = ar1(N_TOY, R_TOY, rng_for(900))
x_white = ar1(N_TOY, 0.0, rng_for(901))

tau_toy = tau_int(x_toy)
check("tau_int on AR(1), r = 0.9: exact 19", abs(tau_toy - 19) < 0.2 * 19, f"got {tau_toy:.2f}")
check("tau_int on white noise: exact 1", abs(tau_int(x_white) - 1) < 0.3, f"got {tau_int(x_white):.2f}")

naive = naive_error(x_toy)
check("naive_error = s / sqrt(N)", np.isclose(naive, np.std(x_toy) / np.sqrt(N_TOY), rtol=1e-3),
      f"got {naive:.5f}")
B_TOY = 2 ** np.arange(0, 11)
curve = blocking_curve(x_toy, B_TOY)
check("blocking at B = 1 equals the naive error", np.isclose(curve[0], naive, rtol=1e-3),
      f"{curve[0]:.5f} vs {naive:.5f}")
exact_err = ar1_exact_error(N_TOY, R_TOY)
check("blocking plateau (B = 1024) = exact error of the mean", abs(curve[-1] / exact_err - 1) < 0.25,
      f"{curve[-1]:.5f} vs exact {exact_err:.5f}")
check("blocked / naive ~ sqrt(tau_int)", abs((curve[-1] / naive) ** 2 / 19 - 1) < 0.5,
      f"(blocked/naive)^2 = {(curve[-1] / naive) ** 2:.1f}")

---
## Problem 1: MCMC and the 2D Ising model — theory

**What I sample.** A configuration $\sigma$ has energy $E(\sigma)=-J\sum_{\langle ij\rangle}\sigma_i\sigma_j$ (each of the $2|\Lambda|=800$ bonds once) and probability $p(\sigma)=e^{-E(\sigma)/T}/Z$. The sum for $Z$ has $2^{400}\approx10^{120}$ terms, so instead I build a Markov chain whose stationary distribution is $p(\sigma)$ and average over it:
$$
\langle m\rangle\approx\frac1K\sum_{k=1}^{K}m\big(\sigma^{(k)}\big),\qquad m(\sigma)=\frac1{|\Lambda|}\sum_i\sigma_i .
$$

**Metropolis from detailed balance.** Propose $\sigma\to\sigma'$ by flipping one site chosen uniformly, so $q(\sigma'|\sigma)=q(\sigma|\sigma')=1/|\Lambda|$ (symmetric). Accept with
$$
A(\sigma'|\sigma)=\min\!\left(1,\frac{p(\sigma')}{p(\sigma)}\right)=\min\!\left(1,e^{-\Delta E/T}\right).
$$
Then $P(\sigma'|\sigma)p(\sigma)=q\,\min\big(p(\sigma),p(\sigma')\big)$ is symmetric in $\sigma\leftrightarrow\sigma'$, so $p$ is stationary; $Z$ cancels in the ratio. Single flips reach every configuration (irreducible) and rejections make the chain aperiodic.

**Local energy change.** Only the four bonds touching site $i$ change. With $h_i=\sum_{j\in\mathcal N(i)}\sigma_j\in\{-4,-2,0,2,4\}$,
$$
\Delta E=E(\sigma')-E(\sigma)=2J\,\sigma_i h_i\in\{-8,-4,0,4,8\}.
$$
Only $\Delta E=4,8$ need a random number, so I precompute $e^{-4/T}$ and $e^{-8/T}$ once per temperature.

**What to compare with.**
- Onsager: $T_c=2/\ln(1+\sqrt2)\approx2.269$.
- Yang (infinite lattice, $T<T_c$): $m_0(T)=\big[1-\sinh^{-4}(2J/T)\big]^{1/8}$, e.g. $m_0(2)\approx0.911$. $m_0=0$ for $T>T_c$.
- **Pitfall:** $E(\sigma)=E(-\sigma)$, so on a finite lattice $\langle m\rangle=0$ exactly. A long enough chain eventually flips between $\pm m$, more easily near $T_c$ and on small lattices. I report $\langle|m|\rangle$ (allowed by the autograder).

**Algorithm.**
```
spins <- initial configuration (all up, or random +-1)
for sweep = 1 .. n_sweeps:
    repeat 400 times:
        pick site (i, j) uniformly
        dE = 2 * s[i,j] * (sum of 4 neighbors, periodic)
        if dE <= 0 or u < exp(-dE / T): flip s[i,j]
    record m = mean(spins)          # one measurement per sweep
```
Compiled with `@njit`, as in Lab 11; I count accepted flips to report the acceptance rate.

**Error bars.** Successive sweeps are correlated, so $s/\sqrt N$ is too small by $\sqrt{\tau_\text{int}}$. I use blocking, and check it against $\tau_\text{int}$ (Setup). Near $T_c$ the autocorrelation time grows (critical slowing down, $\tau\propto L^{z}$, $z\approx2.17$), so the chains there must be the longest.

### Problem 1A — plan
- **Two chains at $T=2$**, same length: all spins up ($k=0$) and random $\pm1$ ($k=1$). Plot $m$ against sweeps for both.
- **Burn-in criterion:** the sweep after which the trace stays inside the equilibrium band (the mean $\pm$ the fluctuation width of the second half of the run).
  - *All up:* starts at $m=1$, close to $m_0(2)\approx0.91$, so I expect a short burn-in (tens of sweeps).
  - *Random:* starts at $m\approx0$ and must coarsen. Domains grow, so I expect it to take hundreds of sweeps to align the $20\times20$ lattice. It can also get stuck in a **stripe** state (two flat domain walls wrapping around the lattice, $m\approx0$) for a long time. If the trace sits near $m\approx0$, look at the lattice.
- **Total steps:** after burn-in, measure $\tau_\text{int}$. Choose the run length so that $N\gtrsim50\,\tau_\text{int}$ (reliable $\tau$) and the blocking error on $\langle m\rangle$ is small compared with what I want to resolve.

### 1A Run and plot

In [ ]:
L = 20
T_C = 2 / np.log(1 + np.sqrt(2))                  # Onsager, ~2.269


def ising_energy(spins):
    """Total energy -J sum_<ij> s_i s_j, each bond once (up and left neighbors, periodic).
    Written independently of the sampler so it can test it. Valid for lattice side >= 3."""
    return -float(np.sum(spins * (np.roll(spins, 1, axis=0) + np.roll(spins, 1, axis=1))))


@njit
def flip_energy(spins, i, j):
    """Energy change if spin (i, j) were flipped: 2 J s_ij * (sum of its 4 periodic neighbors).
    Only reports the cost; does not flip anything."""
    # TODO
    raise NotImplementedError


@njit
def ising_sweeps(spins, T, n_sweeps, rng):
    """n_sweeps Metropolis sweeps (L*L attempted flips each) in place.
    Returns m after every sweep (length n_sweeps + 1, m[0] = start) and the acceptance rate.
    Uses flip_energy. A rejected flip must leave the lattice unchanged (Lab 10, Part 3)."""
    # TODO: precompute exp(-4/T), exp(-8/T); dE = 2 s_ij * (4 periodic neighbors); accept/reject; record m
    raise NotImplementedError


def initial_spins(kind, rng):
    """'up' -> all +1;  'random' -> +1/-1 with probability 1/2 each."""
    # TODO
    raise NotImplementedError


def yang_m0(T):
    """Spontaneous magnetization of the infinite lattice: (1 - sinh(2/T)^-4)^(1/8) below T_c, 0 above."""
    # TODO
    raise NotImplementedError

### Validation of the Ising sampler (run before 1A)
Each test is aimed at a mistake from the Lab 10 checklist; Lab 10's Part 3 sampler had the second and third.

| Test | Checklist item | Catches |
|---|---|---|
| `flip_energy` $=E(\text{after})-E(\text{before})$ at all 400 sites | edges, invariants | missing factor 2, sign error, wrong neighbor or wrap index |
| $3\times3$ lattice vs the exact sum over all $2^9=512$ configurations, $T=1,\,2,\,2.269,\,4,\,1000$ | known answer, limits | rejected flips not undone (looks like $T=\infty$), factor 2 in $\Delta E$ (looks like $2T$), wrong acceptance rule |
| one $m$ per sweep with `m[0]` = start; acceptance $\to0$ as $T\to0$ and $\to1$ as $T\to\infty$ | bookkeeping, limits | recording only accepted moves, miscounted acceptance |

Error bars for the $3\times3$ comparison come from the spread of 8 independent chains, not from the tools being tested. A small $3\times3$ lattice flips between $\pm m$ easily, which is exactly why the comparison uses $\langle|m|\rangle$.

In [ ]:
def exact_ising(n, T):
    """Exact <|m|> on an n x n periodic lattice: Boltzmann-weighted sum over all 2^(n*n) configurations."""
    configs = np.array(list(itertools.product([-1, 1], repeat=n * n))).reshape(-1, n, n)
    E = np.array([ising_energy(s) for s in configs])
    weights = np.exp(-(E - E.min()) / T)             # shift by E.min() to avoid overflow at low T
    return np.sum(weights * np.abs(configs.mean(axis=(1, 2)))) / weights.sum()


# 1. Local energy change vs the independent total energy, at every site (edges included)
s = initial_spins("random", rng_for(910))
E_before, worst, values = ising_energy(s), 0.0, set()
for i in range(L):
    for j in range(L):
        dE = flip_energy(s, i, j)
        s[i, j] *= -1
        worst = max(worst, abs((ising_energy(s) - E_before) - dE))
        s[i, j] *= -1
        values.add(float(dE))
check("flip_energy = E(after) - E(before) at all 400 sites", worst == 0, f"max difference {worst}")
check("dE only takes the values -8, -4, 0, 4, 8", values <= {-8.0, -4.0, 0.0, 4.0, 8.0}, f"saw {sorted(values)}")

# 2. Known answer: 3 x 3 lattice vs exact enumeration, 8 independent chains per temperature
N_CHAINS_V, SWEEPS_V, BURN_V = 8, 20_000, 1_000
for iT, T in enumerate([1.0, 2.0, 2.269, 4.0, 1000.0]):
    means = []
    for c in range(N_CHAINS_V):
        rng = rng_for(920 + 10 * iT + c)
        m, _ = ising_sweeps(rng.choice(np.array([-1, 1]), size=(3, 3)), T, BURN_V + SWEEPS_V, rng)
        means.append(np.abs(m[BURN_V + 1:]).mean())
    mc, err = np.mean(means), np.std(means, ddof=1) / np.sqrt(N_CHAINS_V)
    exact = exact_ising(3, T)
    check(f"3x3 <|m|> at T = {T:g}: exact {exact:.4f}", abs(mc - exact) < max(4 * err, 0.005),
          f"MCMC {mc:.4f} +- {err:.4f}")

# 3. Bookkeeping and limits on the 20 x 20 lattice
m, acc = ising_sweeps(initial_spins("up", rng_for(960)), 2.0, 50, rng_for(961))
check("one m per sweep plus the start, m[0] = 1", len(m) == 51 and m[0] == 1.0, f"len {len(m)}, m[0] = {m[0]}")
m, acc = ising_sweeps(initial_spins("up", rng_for(962)), 0.5, 200, rng_for(963))
check("T -> 0: all-up start stays ordered, almost nothing accepted", m.min() > 0.99 and acc < 1e-3,
      f"min m {m.min():.3f}, acceptance {acc:.1e}")
m, acc = ising_sweeps(initial_spins("up", rng_for(964)), 1e6, 200, rng_for(965))
check("T -> infinity: nearly every flip accepted, m -> 0", acc > 0.99 and abs(m[-50:]).mean() < 0.1,
      f"acceptance {acc:.4f}")

In [ ]:
T_1A = 2.0
SWEEPS_1A = None   # TODO: choose (long enough to see both relax and to measure tau_int)

# TODO: run both starts (k = 0: 'up', k = 1: 'random'); plot m vs sweep for both, with yang_m0(2) as a reference line
# TODO: mark the chosen burn-in of each run; print acceptance rates

In [ ]:
# TODO: after burn-in: autocorrelation of m, tau_int, blocking curve -> error on <m> (or <|m|>) for each start

**Answer (1A):** _TODO_
- **Burn-in, all-up start:** ___ sweeps, because ___.
- **Burn-in, random start:** ___ sweeps, because ___ (stripe state seen? ___).
- **Total steps:** $\tau_\text{int}\approx$ ___ sweeps at $T=2$, so ___ sweeps give $\langle|m|\rangle=$ ___ $\pm$ ___ (vs $m_0(2)=0.911$).

### Problem 1B — strategy (theory)
Temperatures $T=1.0,1.2,\dots,4.0$ (16 values). For each, I decide:

- **Initial configuration:** all up for every $T$. Below $T_c$ it is already close to equilibrium. Above $T_c$ it relaxes quickly (short correlation length), and using one start everywhere keeps the runs comparable. _(Alternative to discuss: anneal, i.e. start each $T$ from the last configuration at the neighboring temperature.)_
- **Burn-in:** from 1A, scaled with $\tau_\text{int}(T)$: discard at least $\sim20\,\tau_\text{int}$, more near $T_c$.
- **Sweeps per chain:** $N\gtrsim50\,\tau_\text{int}(T)$. $\tau$ is largest near $T_c$, so I size the run for the worst case (or per $T$).
- **Number of chains:** several independent chains per $T$ (seeds $100+10i_T+c$). The spread of the chain means, divided by $\sqrt{n_\text{chains}}$, is an independent cross-check of the blocking error, and it reveals chains stuck in different states.
- **Thinning:** none. Measuring $m$ costs nothing next to a sweep, and blocking already handles the correlation (Lecture 18: keep every sweep after burn-in). Thinning would only throw information away.

**What to plot.** $\langle|m|\rangle$ against $T$ with (a) a band for the **standard deviation of $m$** within the chain (thermal fluctuations, which peak near $T_c$) and (b) the **error bar on the mean** from blocking. _(The assignment's wording, "its standard deviation", fits (a). I'll show both and say which is which.)_ Overlay $m_0(T)$ and $T_c$.

**Expectation, sharp transition?** No: on a $20\times20$ lattice the transition is rounded and shifted. The correlation length cannot exceed $L$, so $\langle|m|\rangle$ stays above zero above $T_c$: $\langle|m|\rangle\sim\sqrt{\langle m^2\rangle}=\sqrt{\chi T/|\Lambda|}\propto1/L$ (random-sign patches that don't cancel), and below it the drop is smooth. Other causes to discuss: critical slowing down (too-short chains near $T_c$), sign flips between $\pm m$ if I averaged $m$ instead of $|m|$, and metastable stripe states.

### 1B Temperature scan

In [ ]:
T_GRID = np.round(np.arange(1.0, 4.0 + 1e-9, 0.2), 1)    # 16 temperatures
N_CHAINS = None    # TODO
BURNIN_1B = None   # TODO (per T if needed)
SWEEPS_1B = None   # TODO

# TODO: for each T (index iT) and chain c: rng_for(100 + 10 * iT + c), all-up start, run, drop burn-in
# TODO: store per T: <|m|>, std of m, blocking error, tau_int, acceptance, spread of chain means
m_of_T = None      # 16 values for results.json

In [ ]:
# TODO: plot <|m|> vs T with std band and blocking error bars; yang_m0(T) curve; vertical line at T_C
# TODO (optional): tau_int vs T, to show critical slowing down

**Answer (1B):** _TODO_
- **Strategy used:** initial configuration ___, burn-in ___, sweeps per chain ___, chains ___, thinning ___.
- **Comparison with $T_c$:** ___
- **Sharp transition?** ___ What could be going wrong: ___

---
## Problem 2: Path-integral Monte Carlo at finite temperature — theory

**From $Z$ to a probability over paths.** With $\tau$ the imaginary time, $Z=\mathrm{Tr}\,e^{-\beta\hat H}$ is a path integral over **closed** paths of length $\beta$ (Lecture 13: the trace sets $x_N\equiv x_0$). Slicing it into $N=\beta/\delta\tau$ pieces turns each path into a point in $\mathbb R^N$ with
$$
p(\mathbf x)\propto e^{-S_E(\mathbf x)},\qquad
S_E=\sum_{i=0}^{N-1}\left[\frac{(x_{i+1}-x_i)^2}{2\,\delta\tau}+\delta\tau\,V(x_i)\right].
$$
Each slice is tied to its neighbors by a spring with constant $1/\delta\tau$, and to the origin by $\delta\tau V$. This is the same problem as Problem 1: a ring of $N$ sites instead of a lattice, $x_i$ for $\sigma_i$, and $S_E$ for $E/T$ (Lecture 18).

**Single-slice Metropolis move.** Propose $x_i\to x_i'=x_i+u$ with $u\sim U[-h,h]$ (symmetric). Only the two springs touching $x_i$ and its potential term change:
$$
\Delta S_E=\frac{(x_i'-x_i)\,(x_i'+x_i-x_{i-1}-x_{i+1})}{\delta\tau}+\delta\tau\,\big[V(x_i')-V(x_i)\big],
$$
with indices on the ring ($x_{-1}\equiv x_{N-1}$, $x_N\equiv x_0$). Accept with $\min(1,e^{-\Delta S_E})$. One sweep is $N$ attempted updates. Here $N=10$.

**Estimator.** For each sweep, $f=\frac1N\sum_i x_i^2$; $\langle x^2\rangle_\beta$ is the average of $f$ over the sweeps kept after burn-in.

**Exact result.** The oscillator levels have $\langle n|x^2|n\rangle=n+\tfrac12$ and $E_n=n+\tfrac12$, so
$$
\langle x^2\rangle_\beta=\frac{\sum_n(n+\frac12)e^{-\beta(n+\frac12)}}{\sum_ne^{-\beta(n+\frac12)}}
=-\frac{d}{d\beta}\ln Z,\quad Z=\frac{1}{2\sinh(\beta/2)}
\;\Rightarrow\;
\langle x^2\rangle_\beta=\tfrac12\coth\!\left(\tfrac\beta2\right).
$$
Limits: $\to\tfrac12$ (ground state, $\beta\gg1$) and $\to1/\beta$ (equipartition, $\beta\ll1$). At $\beta=1$: $\tfrac12\coth\tfrac12=1.08198$.

The **discretized** chain has its own exact answer (Creutz–Freedman, Lecture 19 slide 7): with $R=1+\tfrac{\delta\tau^2}2-\delta\tau\sqrt{1+\delta\tau^2/4}$,
$$
\langle x^2\rangle_{\delta\tau}=\frac{1}{2\sqrt{1+\delta\tau^2/4}}\;\frac{1+R^N}{1-R^N}.
$$
I checked that this gives $1.08101$ at $\beta=1$, $\delta\tau=0.1$: $-0.089\%$, the "about 0.09%" quoted in 2B(ii).

**Choosing $h$.** Given its neighbors, $x_i$ is Gaussian with variance $\delta\tau/(2+\delta\tau^2)\approx\delta\tau/2$ (width $\approx0.22$). A hit size of a few times that, $h\approx2\sqrt{\delta\tau}\approx0.63$ (Creutz–Freedman), gives $\sim50\%$ acceptance. I scan $h$ and pick a value inside 40–60%.

**Burn-in.** Cold start $\mathbf x=0$. The trace of $f$ relaxes on the slowest timescale of the chain (below), so I discard a multiple of $\tau_\text{int}$ and confirm by eye on the time series.

**Error bars.** Blocking plateau $\sigma_\text{block}$ vs naive $\sigma_\text{naive}=s/\sqrt N$. Since $\sigma_\text{block}^2\approx\tau_\text{int}\,s^2/N$, I expect $\sigma_\text{block}/\sigma_\text{naive}\approx\sqrt{\tau_\text{int}}$, a consistency check between the two methods.

> **Convention to settle.** Lectures 16 and 19 define $\tau_\text{int}=1+2\sum_{t\ge1}\rho(t)$, so that $\sigma^2=\tau_\text{int}\,\mathrm{Var}/N$. Lab 12 calls that quantity $g$ and says $\tau_\text{int}=g/2$. These differ by a factor 2. I use the lecture definition; _TODO: confirm with the TA / Piazza which one `tau_single` expects_.

### Problem 2A(ii) — why single-slice updates decorrelate slowly (theory)
- One update moves one slice by $|u|\le h\sim\sqrt{\delta\tau}$. The path **center** $\bar x=\frac1N\sum_ix_i$ moves by only $u/N$.
- The springs do not hold $\bar x$ in place: shifting every $x_i$ by the same amount leaves all $(x_{i+1}-x_i)$ unchanged. Only the weak potential term, $\delta\tau\,V$, pulls $\bar x$ back. For the oscillator it adds $\tfrac\beta2\bar x^2$ to $S_E$, so $\bar x$ has width $\sim1/\sqrt\beta$, which is large compared with a single step $\sqrt{\delta\tau}/N$.
- Per sweep ($N$ updates), with an effective step variance $\sigma_u^2\sim\delta\tau$ (accepted moves), $\bar x$ is pulled back by only $\sim\tfrac12\sigma_u^2\,\delta\tau\,\bar x$. The relaxation time is therefore
$$
\tau_{\bar x}\sim\frac{2}{\sigma_u^2\,\delta\tau}\propto\frac1{\delta\tau^2},
$$
i.e. **hundreds of sweeps** at $\delta\tau=0.1$. Since $f=\frac1N\sum x_i^2$ contains $\bar x^2$, $f$ inherits this slow mode. The local moves are tuned to the stiff springs ($1/\delta\tau$), while the slow mode only feels the soft potential, and the mismatch grows as $\delta\tau\to0$.
- _TODO: compare this estimate with the measured `tau_single`._

### 2A(i) Run, time series, autocorrelation, blocking

In [ ]:
DTAU = 0.1
BETA_2A = 1.0
N_SLICES = int(round(BETA_2A / DTAU))     # 10


def action(x, dtau):
    """Full discretized action S_E of the closed path x (eq. 4, V = x^2 / 2), with x_N = x_0.
    Written independently of the sampler so it can test it."""
    return float(np.sum((np.roll(x, -1) - x) ** 2 / (2 * dtau) + dtau * 0.5 * x**2))


@njit
def delta_action(x, i, x_new, dtau):
    """Change in S_E when only slice i moves from x[i] to x_new (local formula, periodic neighbors).
    Only reports the change; does not move anything."""
    # TODO: (x_new - x_i)(x_new + x_i - x_{i-1} - x_{i+1}) / dtau + dtau (V(x_new) - V(x_i))
    raise NotImplementedError


@njit
def pimc_sweeps(x, dtau, h, n_sweeps, rng, h_shift=0.0):
    """n_sweeps sweeps in place on the closed path x (length N).
    Each sweep: N single-slice updates x_i -> x_i + U[-h, h];
    if h_shift > 0, then one whole-path shift x -> x + U[-h_shift, h_shift] (Problem 2B).
    Uses delta_action. Both proposals are symmetric, so no Hastings factor appears (Lab 10, Part 2).
    Returns f = mean(x^2) after every sweep (length n_sweeps; rejected updates count),
    the single-slice acceptance and the shift acceptance."""
    # TODO: dS = (x' - x)(x' + x - x_{i-1} - x_{i+1}) / dtau + dtau (V(x') - V(x)), periodic indices
    # TODO: shift move: dS = dtau * sum(V(x_i + d) - V(x_i))  (springs unchanged)
    raise NotImplementedError


def x2_exact(beta):
    """Continuum result 1/2 coth(beta/2)."""
    # TODO
    raise NotImplementedError


def x2_exact_discrete(beta, dtau):
    """Exact <x^2> of the discretized chain (Creutz-Freedman), to separate statistics from discretization."""
    # TODO
    raise NotImplementedError

### Validation of the path-integral sampler (run before 2A)

| Test | Checklist item | Catches |
|---|---|---|
| `delta_action` $=S_E(\mathbf x')-S_E(\mathbf x)$ for every slice, including $i=0$ and $N-1$ | edges, invariants | wrong spring term, ring not closed, missing $1/\delta\tau$ |
| shift: $S_E(\mathbf x+d)-S_E(\mathbf x)=\beta\,(d\,\bar x+d^2/2)$ | invariants | the 2B claim that the springs do not change |
| `x2_exact(1)` $=1.081977$, `x2_exact_discrete(1, 0.1)` $=1.081010$; discrete $\to$ continuum as $\delta\tau\to0$; $\beta\to0$ gives $1/\beta$ | known answer, limits | the reference values themselves |
| one $f$ per sweep; acceptance $\to1$ as $h\to0$ and $\to0$ as $h\to\infty$ | bookkeeping, limits | recording only accepted moves, miscounted acceptance |
| $\langle x^2\rangle$ from 8 chains vs `x2_exact_discrete(1, 0.1)`, single-slice moves and (bonus) both moves | known answer | wrong acceptance rule; a Hastings factor where none belongs |

The chain comparison is against the **discretized** exact value, so a correct sampler agrees within its error bar, not just to within the 0.09 % time-step shift.

In [ ]:
TEST_SHIFT_MOVE = True     # set False if I skip bonus 2B

# 1. Local action change vs the independent full action, every slice (ring edges included)
rng = rng_for(970)
x = rng.normal(size=N_SLICES)
worst = 0.0
for i in range(N_SLICES):
    x_new = x[i] + rng.uniform(-1, 1)
    x_moved = x.copy()
    x_moved[i] = x_new
    worst = max(worst, abs((action(x_moved, DTAU) - action(x, DTAU)) - delta_action(x, i, x_new, DTAU)))
check("delta_action = S(x') - S(x) for every slice", worst < 1e-12, f"max difference {worst:.1e}")
d = 0.37
check("shift move: S(x + d) - S(x) = beta (d xbar + d^2 / 2)",
      np.isclose(action(x + d, DTAU) - action(x, DTAU), BETA_2A * (d * x.mean() + d**2 / 2), atol=1e-12))

# 2. Reference values and their limits
check("x2_exact(1) = 1/2 coth(1/2) = 1.081977", np.isclose(x2_exact(1.0), 1.0819767, atol=1e-6))
check("x2_exact_discrete(1, 0.1) = 1.081010 (-0.089 %)", np.isclose(x2_exact_discrete(1.0, 0.1), 1.0810096, atol=1e-6))
check("discrete -> continuum as dtau -> 0", np.isclose(x2_exact_discrete(1.0, 0.001), x2_exact(1.0), rtol=1e-5))
check("beta -> 0: classical 1/beta", np.isclose(0.01 * x2_exact(0.01), 1.0, rtol=1e-4))

# 3. Bookkeeping and acceptance limits
f, acc, _ = pimc_sweeps(np.zeros(N_SLICES), DTAU, 0.6, 100, rng_for(971))
check("one f per sweep", len(f) == 100, f"len {len(f)}")
_, acc_small, _ = pimc_sweeps(np.zeros(N_SLICES), DTAU, 1e-4, 200, rng_for(972))
_, acc_large, _ = pimc_sweeps(np.zeros(N_SLICES), DTAU, 100.0, 200, rng_for(973))
check("acceptance -> 1 as h -> 0 and -> 0 as h -> infinity", acc_small > 0.99 and acc_large < 0.02,
      f"{acc_small:.3f}, {acc_large:.3f}")

# 4. Known answer: <x^2> at beta = 1 from 8 independent chains vs the discretized exact value
exact = x2_exact_discrete(BETA_2A, DTAU)
for label, h_shift, k0 in [("single-slice moves", 0.0, 980), ("both moves", 2.0, 990)]:
    if h_shift > 0 and not TEST_SHIFT_MOVE:
        continue
    means = []
    for c in range(8):
        f, _, _ = pimc_sweeps(np.zeros(N_SLICES), DTAU, 0.63, 42_000, rng_for(k0 + c), h_shift)
        means.append(f[2_000:].mean())
    mc, err = np.mean(means), np.std(means, ddof=1) / np.sqrt(8)
    check(f"<x^2> at beta = 1, {label}: exact (dtau = 0.1) {exact:.4f}", abs(mc - exact) < 4 * err,
          f"MCMC {mc:.4f} +- {err:.4f}")

In [ ]:
# TODO: scan h (short runs) -> acceptance vs h; pick H_2A with 40-60 % acceptance
H_2A = None

In [ ]:
SWEEPS_2A = None    # TODO: >= 50,000 kept sweeps
BURNIN_2A = None    # TODO: justify from the time series

# TODO: run with rng_for(200), cold start x = 0
# TODO: plot time series of f (mark burn-in) and its autocorrelation rho(t)
# TODO: tau_single = tau_int(f), blocking curve -> x2_err_single; naive error -> x2_err_naive
# TODO: compare x2_single with x2_exact(1) and x2_exact_discrete(1, DTAU)

**Answer (2A(i)):** _TODO_
- $h=$ ___, acceptance ___ %.
- Burn-in ___ sweeps, because ___.
- $\tau_\text{int}=$ ___ sweeps; $\langle x^2\rangle_\beta=$ ___ $\pm$ ___ (blocking) vs naive $\pm$ ___ (ratio ___ $\approx\sqrt{\tau_\text{int}}$?).
- Exact $1.08198$: difference ___ $\sigma$.

**Answer (2A(ii)):** _TODO: summarize the theory above in 2–3 lines and compare with the measured $\tau_\text{int}$._

---
## Problem 2B (bonus): the whole-path shift move — theory

**Which terms change.** $x_i\to x_i+d$ for all $i$ leaves every difference $x_{i+1}-x_i$, and so the whole kinetic (spring) part of $S_E$, unchanged. Only the potential part changes:
$$
\Delta S_E=\delta\tau\sum_i\big[V(x_i+d)-V(x_i)\big]\;\overset{V=x^2/2}{=}\;\delta\tau\Big(d\sum_ix_i+\tfrac N2d^2\Big)=\beta\Big(d\,\bar x+\tfrac{d^2}{2}\Big).
$$
It costs $O(N)$ to evaluate, the same as one sweep of single-slice updates.

**Why it is a valid Metropolis move.** The proposal is symmetric ($d$ and $-d$ are equally likely, so $q(\mathbf x'|\mathbf x)=q(\mathbf x|\mathbf x')$). Accepting with $\min(1,e^{-\Delta S_E})$ therefore satisfies detailed balance with respect to $p\propto e^{-S_E}$, so this move alone keeps $p$ stationary. A sweep followed by a shift is a composition of two kernels that each preserve $p$, so it preserves $p$ too. The single-slice moves already make the chain irreducible and aperiodic.

**Why it helps.** It moves $\bar x$ directly, by $O(1/\sqrt\beta)$ in one step instead of $\sqrt{\delta\tau}/N$, so the slow mode from 2A(ii) decorrelates in a few sweeps. **Choosing $h'$:** $\bar x$ has width $1/\sqrt\beta$, so $h'\sim$ a few $/\sqrt\beta$ gives a reasonable shift acceptance (scan it, as for $h$).

**2B(ii) plan.** $\beta=0.5,1,2,4$ at $\delta\tau=0.1$, so $N=5,10,20,40$, with both moves (re-tune $h'$ for each $\beta$). Plot the results with error bars against $\tfrac12\coth(\beta/2)$. **Statistics or discretization?** At $\beta=1$ the discretization shift is $-0.09\%\approx-0.001$. If my error bar is larger than $0.001$, the result is statistics-limited; to resolve the bias the error must be well below it. A direct test is to compare with `x2_exact_discrete`.

### 2B(i) Both moves at $\beta=1$

In [ ]:
H_SHIFT = None      # TODO: scan, like h
SWEEPS_2B = None    # TODO

# TODO: run with rng_for(300), same h as 2A, plus shift move
# TODO: same analysis as 2A(i): time series, rho(t), tau_both, blocking error x2_err_both
# TODO: side-by-side autocorrelation (single vs both moves)

**Answer (2B(i)):** _TODO_
- Terms of $S_E$ that change and why the move is valid: see theory.
- $\tau_\text{int}$: ___ (single) → ___ (both); error bar ___ → ___; agreement with exact: ___.

### 2B(ii) $\langle x^2\rangle_\beta$ vs $\beta$

In [ ]:
BETAS = [0.5, 1.0, 2.0, 4.0]
n_beta, x2_beta, x2_err_beta = [], [], []

# TODO: for each beta (index ib): N = beta / DTAU, rng_for(400 + ib), both moves, burn-in, blocking error
# TODO: plot points with error bars vs beta, with x2_exact(beta) curve (and x2_exact_discrete)

**Answer (2B(ii)):** _TODO_
- At $\beta=1$: error bar ___ vs discretization shift $\approx0.001$, so the result is limited by ___.

---
## Autograded results

In [ ]:
failed = [name for name, ok in VALIDATION if not ok]
print(f"Validation: {len(VALIDATION) - len(failed)}/{len(VALIDATION)} passed" + (f"; FAILED: {failed}" if failed else ""))

results = {
    "tsn": TSN,
    "seed": SEED,
    "m": [float(v) for v in m_of_T],                                   # 1B: 16 values
    "h": float(H_2A), "acceptance": float(acceptance_2A), "n_sweeps_single": int(n_single),
    "x2_single": float(x2_single), "x2_err_single": float(x2_err_single),
    "x2_err_naive": float(x2_err_naive), "tau_single": float(tau_single),
}
# Problem 2B (bonus): delete any part I skip
results.update({
    "h_shift": float(H_SHIFT), "n_sweeps_both": int(n_both),
    "x2_both": float(x2_both), "x2_err_both": float(x2_err_both), "tau_both": float(tau_both),
    "n_sweeps_beta": [int(v) for v in n_beta],
    "x2_beta": [float(v) for v in x2_beta],
    "x2_err_beta": [float(v) for v in x2_err_beta],
})
with open("results.json", "w") as f:
    json.dump(results, f, indent=1)

---
## AI use appendix
_TODO_
- **Tool:** Claude Code (Anthropic). Conversation: https://claude.ai/code/session_0139N3DcDBvXoCicEtTrNrj2
- **Purpose:** ___
- **Prompts:** ___
- **What it got wrong and how I caught it:** see the audit log below.

**Audit log** (Lab 10 format). One row per mistake, mine or the AI's. After each fix I rerun **all** validation cells, one fix at a time, because a second mistake can hide behind the first.

| # | Function / cell | Symptom | Test that exposed it | Fix | Would I have caught it without that test? |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |